# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [2]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [3]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



Google API Key exists and begins AI
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [4]:
# Connect to client libraries

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [5]:
models = ["gemini-2.5-flash-lite", "llama-3.3-70b-versatile", "llama3.2:latest", "qwen/qwen3-coder-free",]

clients = {"gemini-2.5-flash-lite": gemini, "llama-3.3-70b-versatile": groq, "llama3.2:latest": ollama, "qwen/qwen3-coder-free": openrouter}

In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.22631',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [7]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


## And now, on with the main task

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [11]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [18]:
def compile_and_run():
    import os
    import subprocess

    # Compile
    subprocess.run(
        ["g++", "main.cpp", "-o", "main"],
        check=True
    )

    # Windows vs Linux/Mac
    executable = "main.exe" if os.name == "nt" else "./main"

    outputs = []

    for _ in range(3):
        result = subprocess.run(
            [executable],
            text=True,
            capture_output=True,
            check=True
        )

        outputs.append(result.stdout)

    return "\n".join(outputs)

In [32]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [33]:
compile_and_run()

'Result: 3.14159265609\nExecution Time: 1.34329 seconds\n\nResult: 3.14159265609\nExecution Time: 1.41027 seconds\n\nResult: 3.14159265609\nExecution Time: 1.34763 seconds\n'

In [34]:
#llama-3.3-70b-versatile

104.374508/1.34329

77.70065138577671

In [ ]:
#Gemini-2.5-flash-lite

104.374508/1.037582

100.59398486095557

Qwen 3 Coder Free: Fail  
Gemini-2.5-flash-lite: 1.037582 seconds
Groq llama-3.3-70b-versatile: 1.34329 seconds


In [35]:
print(f"""
In today's experiments, the performance speedups were:

2nd place: Groq Llama-3.3-70b-versatile: {104.374508/1.34329:.0f}X speedup
1st place: Gemini 2.5 Flash Lite: {104.374508/1.037582:.0f}X speedup
""")


In today's experiments, the performance speedups were:

2nd place: Groq Llama-3.3-70b-versatile: 78X speedup
1st place: Gemini 2.5 Flash Lite: 101X speedup

